# Capture rate and BST within stages

Retained capture-rate exploration from `explore_corr.ipynb`, using corrected local data. Capture rate is generally unavailable at announcement time; this remains descriptive research, not prediction input.

In [1]:
# Resolve project root before package imports. Works from root or notebooks/.
import os
import sys
from pathlib import Path

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file() and (p / "pokeml").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch this notebook inside the ml_pokemon project")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if sys.version_info[:2] != (3, 14):
    raise RuntimeError("Select project Python 3.14.8 kernel (.pokemon)")
print(f"Project: {PROJECT_ROOT} | Python: {sys.version.split()[0]} | Kernel: {sys.executable}")


Project: /Users/pano/Library/Mobile Documents/com~apple~CloudDocs/coding/portfolio/ml_pokemon | Python: 3.14.8 | Kernel: /Users/pano/venv/pokemon/.pokemon/bin/python


In [2]:
import pandas as pd
import numpy as np
from scipy import stats
from IPython.display import display
from pokeml.eda.snapshot import load_analysis_snapshot
from pokeml.data.validation import validate_pkdx

minimal_df = load_analysis_snapshot()
df = pd.read_csv("datasets/pkdx_clean.csv")
validate_pkdx(df, full=True)
# Capture rate belongs to clean data; join by stable identity, never row position.
minimal_df = minimal_df.merge(df[["id", "capture_rate"]], on="id", how="left", validate="one_to_one")
assert minimal_df.capture_rate.notna().all()
display(minimal_df[["name", "stage", "rarity", "total_stats", "capture_rate"]].head())

,name,stage,rarity,total_stats,capture_rate
0,bulbasaur,s1c3,regular,318,45
1,ivysaur,s2c3,regular,405,45
2,venusaur,s3c3,regular,525,45
3,charmander,s1c3,regular,309,45
4,charmeleon,s2c3,regular,405,45


## Original stage-wise Pearson analysis

Pearson r measures linear association. A small p-value is evidence under statistical assumptions, not a measure of trustworthiness or causation. Multiple stage comparisons and related families limit interpretation. Keep groups with at least ten complete observations; constant variables cannot yield a correlation.

In [3]:
# Let check how the stage of a pokemon, its bst and capture rate are related.
# If r, which means to correlation, is high, both + or -, it points to how correlated its bst is to capture rate.
# p points to how trustworthy the correlation is. p = 0 -> very good (Less that 0.05 points to real correlation going on)

for stage in minimal_df['stage'].dropna().unique():
    subset = minimal_df[minimal_df['stage'] == stage]
    if len(subset) < 10:
        continue
    r, p = stats.pearsonr(subset['capture_rate'], subset['total_stats'])
    print(f"{stage:7s} | n={len(subset):3d} | r={r:+.3f} | p={p:.2f}")


# Present original statistics together for comparison.
records = []
for stage, subset in minimal_df.groupby("stage", observed=True):
    values = subset[["capture_rate", "total_stats"]].dropna()
    if len(values) < 10 or (values.nunique() < 2).any():
        continue
    r, p = stats.pearsonr(values.capture_rate, values.total_stats)
    records.append({"stage": stage, "n": len(values), "pearson_r": r, "p_value": p})
display(pd.DataFrame(records).round(4))

s1c3    | n=117 | r=-0.528 | p=0.00
s2c3    | n=119 | r=-0.641 | p=0.00
s3c3    | n=127 | r=-0.210 | p=0.02
s1c2    | n=240 | r=-0.572 | p=0.00
s2c2    | n=267 | r=-0.361 | p=0.00
single  | n=212 | r=-0.532 | p=0.00


,stage,n,pearson_r,p_value
0,s1c2,240,-0.5723,0.0000
1,s1c3,117,-0.5283,0.0000
2,s2c2,267,-0.3610,0.0000
3,s2c3,119,-0.6413,0.0000
4,s3c3,127,-0.2099,0.0179
5,single,212,-0.5319,0.0000


## Historical category-normalisation attempt

Current clean dataset already provides `rarity`. Original attempt below wrote baby labels into a separate column and modified input in place. Preserve as historical code; do not execute.

```python
def change_cats(database):

    leg_df = database['category'].isin(['legendary', 'mythical'])
    database.loc[leg_df, 'category'] = 'legendary'
    baby_df = database['category'].isin(['baby'])
    database.loc[baby_df, 'baby'] = 'regular'
    caca = database.drop(columns=['baby'], axis=1)

    return caca
change_df = change_cats(minimal_df)
change_df[140:152]
```

## Interpretation

Stage-wise association can still reflect rarity, families and generation composition. Capture difficulty may itself be designed alongside strength. Correlation does not demonstrate an announcement-time predictor.